# Lesson 12 - 使用代理速記本減少聊天歷史記錄

此筆記本展示了如何使用 Microsoft Agent Framework 在長對話中管理上下文。隨著對話增長，令牌數量也會增加 — 最終超出模型的上下文視窗。我們通過**上下文總結模式**和用於持久記憶的**代理速記本**來解決這個問題。

## 你將學到：
1. **為什麼上下文管理很重要**：了解令牌限制和上下文視窗
2. **上下文感知代理**：建立能管理自身對話上下文的代理
3. **上下文總結模式**：利用工具濃縮對話歷史
4. **代理速記本**：可持續保存記憶以應對上下文縮減

## 前置條件：
- 已設定 Azure OpenAI 並配置環境變量
- 了解先前課程中的基本代理概念


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 為什麼上下文管理很重要

每個大型語言模型（LLM）都有一個有限的**上下文窗口** — 即它在單次請求中能處理的最大標記數。隨著多輪對話的進行：

- **標記數會隨每個用戶訊息和助理回覆線性增加**。
- **提示標記佔據成本主導**，因為整個歷史記錄每輪都會重新傳送。
- 最終對話會**超出上下文窗口**，模型要麼會截斷，要麼會報錯。

### 管理上下文的策略

| 策略 | 運作方式 | 折衷 |
|---|---|---|
| **截斷** | 丟棄最舊的訊息 | 失去早期上下文 |
| **摘要** | 將舊訊息濃縮成摘要 | 會失去部分細節，但保留重點 |
| **草稿板 / 外部記憶** | 將關鍵事實存放在對話外 | 需要調用工具，但可維持任何縮減後的狀態 |

在本筆記本中，我們結合了**摘要**和**草稿板工具**，讓代理能在對話歷史被濃縮時仍保持連貫性。


## 創建具情境感知的代理


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模擬長時間對話

讓我們逐步進行多輪對話，看看上下文如何累積。代理人應該保留關鍵細節（偏好、預算、旅行日期）於各輪對話中，並展現連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

請注意代理如何保留早期對話的上下文 — 它了解日本、壽司、寺廟、攝影、3000美元的預算、獨自旅行，以及四月的時間範圍。在短對話中這運作良好，但隨著對話增長，重新發送完整歷史會變得昂貴。

讓我們繼續更多回合的對話，看看上下文如何累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Context Summarization Pattern

隨著對話增長，我們可以使用**摘要工具**將累積的上下文濃縮成簡潔的格式。代理會呼叫此工具以記錄關鍵偏好，即使較舊的訊息被刪除，重要資訊仍能被保留。

此模式是更複雜歷史縮減的基礎建構：
1. 代理從對話中識別關鍵事實
2. 呼叫摘要工具來保存這些資訊
3. 較舊的訊息可以安全地移除，因為摘要已抓取重要內容

以下我們定義了代理可以呼叫的 `summarize_preferences` 工具，用以記錄所學得的簡明摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 摘要

在本課程中，你學會了如何使用 Microsoft Agent Framework 管理長時間運行的代理對話中的上下文：

### 主要概念
- **上下文視窗是有限的** — 對話歷史中的每個令牌都會產生費用並計入限制。
- **摘要工具** 讓代理能將累積的上下文壓縮成簡潔的摘要，減少令牌使用同時保留重要資訊。
- **代理草稿板** 提供持久的外部記憶，即使對話縮減也能保存。

### 你所建立的內容
- 一個 **具備上下文感知的代理** ，能跨多輪對話維持連貫性
- 一個 **摘要工具** (`summarize_preferences`) ，用以以簡潔格式記錄重要用戶細節
- 一個展示上下文保留與變更處理的 **多輪對話**

### 真實世界的應用
- **客戶服務機器人**：在長時間支援會話中記住偏好
- **個人助理**：追蹤進行中的專案，無需重複解釋上下文
- **教育導師**：在多次互動中維持學生進度

### 後續步驟
- 實作具備檔案持久化的完整草稿板工具
- 在摘要後加入自動歷史截斷
- 結合向量資料庫以實現語意記憶搜尋
- 建立能在數天後恢復對話、保有完整上下文的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所提供的翻譯工具完成。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於重要資訊，建議採用專業人工翻譯。我們不對因使用此翻譯而產生的任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
